In [ ]:
# ============================================================================
# PART 1: INSTALLATIONS AND IMPORTS
# ============================================================================

# Install required packages
!pip install prophet xgboost scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
import pickle
import warnings
from datetime import datetime, timedelta
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
from prophet import Prophet

warnings.filterwarnings('ignore')

# ============================================================================
# PART 2: DATA PREPROCESSING PIPELINE
# ============================================================================

class DataPreprocessor:
    """Handles all data preprocessing operations"""

    def __init__(self):
        self.label_encoders = {}
        self.categorical_cols = ['State', 'District', 'Market', 'Commodity']

    def load_and_preprocess(self, filepath):
        """Load and preprocess the dataset"""
        print("Loading dataset...")
        df = pd.read_csv(filepath)

        # Remove specified columns
        columns_to_drop = ['Variety', 'Grade', 'Commodity_Code']
        existing_drops = [col for col in columns_to_drop if col in df.columns]
        df = df.drop(columns=existing_drops)

        # Convert Arrival_Date to datetime
        print("Processing dates...")
        df['Arrival_Date'] = pd.to_datetime(df['Arrival_Date'], format='%d/%m/%Y')
        df['Year'] = df['Arrival_Date'].dt.year
        df['Month'] = df['Arrival_Date'].dt.month
        df['Day'] = df['Arrival_Date'].dt.day
        df['Day_of_Week'] = df['Arrival_Date'].dt.dayofweek
        df['Week_of_Year'] = df['Arrival_Date'].dt.isocalendar().week

        # Encode categorical variables
        print("Encoding categorical variables...")
        for col in self.categorical_cols:
            if col in df.columns:
                self.label_encoders[col] = LabelEncoder()
                df[col] = self.label_encoders[col].fit_transform(df[col].astype(str))

        # Impute missing values in price columns
        print("Imputing missing values...")
        price_cols = ['Min_Price', 'Max_Price', 'Modal_Price']
        df[price_cols] = df[price_cols].ffill()

        for col in price_cols:
            if df[col].isnull().any():
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)

        # Sort by date for time-series operations
        df = df.sort_values(by='Arrival_Date').reset_index(drop=True)

        # Create lag and moving average features
        print("Creating time-series features...")
        df['Modal_Price_Lag_1'] = df.groupby(['State', 'District', 'Market', 'Commodity'])['Modal_Price'].shift(1)
        df['Modal_Price_Lag_7'] = df.groupby(['State', 'District', 'Market', 'Commodity'])['Modal_Price'].shift(7)
        df['Modal_Price_Lag_30'] = df.groupby(['State', 'District', 'Market', 'Commodity'])['Modal_Price'].shift(30)

        df['Modal_Price_MA_7'] = df.groupby(['State', 'District', 'Market', 'Commodity'])['Modal_Price'].transform(
            lambda x: x.rolling(window=7, min_periods=1).mean()
        )
        df['Modal_Price_MA_30'] = df.groupby(['State', 'District', 'Market', 'Commodity'])['Modal_Price'].transform(
            lambda x: x.rolling(window=30, min_periods=1).mean()
        )

        # Fill NaN values created by lag features with forward fill then backward fill
        lag_cols = ['Modal_Price_Lag_1', 'Modal_Price_Lag_7', 'Modal_Price_Lag_30',
                    'Modal_Price_MA_7', 'Modal_Price_MA_30']
        df[lag_cols] = df[lag_cols].ffill().bfill()

        # Add time index for TFT
        df['time_idx'] = (df['Arrival_Date'] - df['Arrival_Date'].min()).dt.days

        print(f"Preprocessing complete! Dataset shape: {df.shape}")
        return df

    def save_encoders(self, filepath='label_encoders.pkl'):
        """Save label encoders for later use"""
        with open(filepath, 'wb') as f:
            pickle.dump(self.label_encoders, f)
        print(f"Label encoders saved to {filepath}")

    def load_encoders(self, filepath='label_encoders.pkl'):
        """Load saved label encoders"""
        with open(filepath, 'rb') as f:
            self.label_encoders = pickle.load(f)
        print(f"Label encoders loaded from {filepath}")

# ============================================================================
# PART 3: XGBOOST MODEL TRAINING
# ============================================================================

class XGBoostPricePredictor:
    """XGBoost model for price prediction"""

    def __init__(self):
        self.model = None
        self.feature_columns = None

    def train(self, df, test_size=0.2, random_state=42):
        """Train XGBoost model"""
        print("\n" + "="*50)
        print("Training XGBoost Model")
        print("="*50)

        # Prepare features
        feature_cols = ['State', 'District', 'Market', 'Commodity', 'Min_Price', 'Max_Price',
                       'Year', 'Month', 'Day', 'Day_of_Week', 'Week_of_Year',
                       'Modal_Price_Lag_1', 'Modal_Price_Lag_7', 'Modal_Price_Lag_30',
                       'Modal_Price_MA_7', 'Modal_Price_MA_30']

        self.feature_columns = feature_cols
        X = df[feature_cols]
        y = df['Modal_Price']

        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, shuffle=False
        )

        # Train model
        print(f"Training on {len(X_train)} samples...")
        self.model = xgb.XGBRegressor(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=7,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=random_state,
            n_jobs=-1
        )

        self.model.fit(X_train, y_train,
                      eval_set=[(X_test, y_test)],
                      verbose=False)

        # Evaluate
        y_pred = self.model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        print(f"\nXGBoost Model Performance:")
        print(f"MAE: {mae:.2f}")
        print(f"RMSE: {rmse:.2f}")
        print(f"R² Score: {r2:.4f}")

        return self.model

    def save_model(self, filepath='xgboost_model.pkl'):
        """Save trained model"""
        with open(filepath, 'wb') as f:
            pickle.dump({'model': self.model, 'features': self.feature_columns}, f)
        print(f"\nXGBoost model saved to {filepath}")

    def load_model(self, filepath='xgboost_model.pkl'):
        """Load saved model"""
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
            self.model = data['model']
            self.feature_columns = data['features']
        print(f"XGBoost model loaded from {filepath}")

# ============================================================================
# PART 4: PROPHET MODEL TRAINING
# ============================================================================

class ProphetPricePredictor:
    """Prophet model for price prediction"""

    def __init__(self):
        self.models = {}  # Store separate models for each commodity-market combination
        self.model_metadata = {}

    def train(self, df, max_models=50):
        """Train Prophet models for different commodity-market combinations"""
        print("\n" + "="*50)
        print("Training Prophet Models")
        print("="*50)

        # Create a combination identifier
        df['combo_id'] = (df['State'].astype(str) + '_' +
                         df['District'].astype(str) + '_' +
                         df['Market'].astype(str) + '_' +
                         df['Commodity'].astype(str))

        # Get top combinations by data volume
        combo_counts = df['combo_id'].value_counts().head(max_models)
        print(f"Training models for top {len(combo_counts)} commodity-market combinations...")

        all_predictions = []
        all_actuals = []

        for idx, (combo_id, count) in enumerate(combo_counts.items(), 1):
            if count < 30:  # Need at least 30 data points
                continue

            print(f"\n[{idx}/{len(combo_counts)}] Training model for: {combo_id} ({count} records)")

            # Filter data for this combination
            combo_data = df[df['combo_id'] == combo_id].copy()
            combo_data = combo_data.sort_values('Arrival_Date')

            # Prepare data in Prophet format (ds, y)
            prophet_df = pd.DataFrame({
                'ds': combo_data['Arrival_Date'],
                'y': combo_data['Modal_Price']
            })

            # Add regressors
            prophet_df['min_price'] = combo_data['Min_Price'].values
            prophet_df['max_price'] = combo_data['Max_Price'].values
            prophet_df['month'] = combo_data['Month'].values
            prophet_df['day_of_week'] = combo_data['Day_of_Week'].values

            # Split into train and test
            train_size = int(len(prophet_df) * 0.8)
            train_df = prophet_df.iloc[:train_size]
            test_df = prophet_df.iloc[train_size:]

            try:
                # Initialize and configure Prophet
                model = Prophet(
                    yearly_seasonality=True,
                    weekly_seasonality=True,
                    daily_seasonality=False,
                    seasonality_mode='multiplicative',
                    changepoint_prior_scale=0.05,
                )

                # Add regressors
                model.add_regressor('min_price')
                model.add_regressor('max_price')
                model.add_regressor('month')
                model.add_regressor('day_of_week')

                # Fit model (suppress output)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    model.fit(train_df)

                # Make predictions on test set
                if len(test_df) > 0:
                    forecast = model.predict(test_df)
                    predictions = forecast['yhat'].values
                    actuals = test_df['y'].values

                    all_predictions.extend(predictions)
                    all_actuals.extend(actuals)

                    mae = mean_absolute_error(actuals, predictions)
                    print(f"  ✓ MAE: {mae:.2f}")

                # Store model and metadata
                self.models[combo_id] = model
                self.model_metadata[combo_id] = {
                    'last_date': combo_data['Arrival_Date'].max(),
                    'last_price': combo_data['Modal_Price'].iloc[-1],
                    'avg_price': combo_data['Modal_Price'].mean(),
                    'count': count
                }

            except Exception as e:
                print(f"  ✗ Error training model: {str(e)[:50]}")
                continue

        # Overall performance
        if len(all_predictions) > 0:
            overall_mae = mean_absolute_error(all_actuals, all_predictions)
            overall_rmse = np.sqrt(mean_squared_error(all_actuals, all_predictions))
            overall_r2 = r2_score(all_actuals, all_predictions)

            print(f"\n{'='*50}")
            print(f"Overall Prophet Model Performance:")
            print(f"Models trained: {len(self.models)}")
            print(f"MAE: {overall_mae:.2f}")
            print(f"RMSE: {overall_rmse:.2f}")
            print(f"R² Score: {overall_r2:.4f}")
            print(f"{'='*50}")

        return self.models

    def save_model(self, filepath='prophet_models.pkl'):
        """Save trained models"""
        with open(filepath, 'wb') as f:
            pickle.dump({
                'models': self.models,
                'metadata': self.model_metadata
            }, f)
        print(f"\nProphet models saved to {filepath}")

    def load_model(self, filepath='prophet_models.pkl'):
        """Load saved models"""
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
            self.models = data['models']
            self.model_metadata = data['metadata']
        print(f"Prophet models loaded from {filepath} ({len(self.models)} models)")

    def predict(self, combo_id, pred_date, min_price, max_price, month, day_of_week):
        """Make prediction using Prophet model"""
        if combo_id not in self.models:
            return None

        model = self.models[combo_id]

        # Prepare future dataframe
        future_df = pd.DataFrame({
            'ds': [pred_date],
            'min_price': [min_price],
            'max_price': [max_price],
            'month': [month],
            'day_of_week': [day_of_week]
        })

        # Make prediction
        forecast = model.predict(future_df)
        return forecast['yhat'].values[0]

# ============================================================================
# PART 5: PREDICTION SYSTEM
# ============================================================================

class PricePredictionSystem:
    """Complete prediction system with dynamic feature engineering"""

    def __init__(self):
        self.preprocessor = DataPreprocessor()
        self.xgb_predictor = XGBoostPricePredictor()
        self.prophet_predictor = ProphetPricePredictor()
        self.historical_data = None

    def load_models(self):
        """Load all saved models and encoders"""
        self.preprocessor.load_encoders('label_encoders.pkl')
        self.xgb_predictor.load_model('xgboost_model.pkl')
        self.prophet_predictor.load_model('prophet_models.pkl')
        print("\nAll models loaded successfully!")

    def set_historical_data(self, df):
        """Set historical data for feature engineering"""
        self.historical_data = df

    def predict(self, state, district, market, commodity, date, model_type='xgboost'):
        """
        Make prediction based on user inputs

        Parameters:
        -----------
        state : str
            State name
        district : str
            District name
        market : str
            Market name
        commodity : str
            Commodity name
        date : str
            Date in format 'YYYY-MM-DD' or 'DD/MM/YYYY'
        model_type : str
            'xgboost' or 'prophet'

        Returns:
        --------
        float : Predicted modal price
        """
        print(f"\n{'='*60}")
        print(f"Making Prediction using {model_type.upper()}")
        print(f"{'='*60}")
        print(f"State: {state}")
        print(f"District: {district}")
        print(f"Market: {market}")
        print(f"Commodity: {commodity}")
        print(f"Date: {date}")
        print(f"{'='*60}\n")

        # Parse date
        try:
            if '/' in date:
                pred_date = pd.to_datetime(date, format='%d/%m/%Y')
            else:
                pred_date = pd.to_datetime(date)
        except:
            raise ValueError("Invalid date format. Use 'YYYY-MM-DD' or 'DD/MM/YYYY'")

        # Encode inputs
        try:
            state_enc = self.preprocessor.label_encoders['State'].transform([state])[0]
            district_enc = self.preprocessor.label_encoders['District'].transform([district])[0]
            market_enc = self.preprocessor.label_encoders['Market'].transform([market])[0]
            commodity_enc = self.preprocessor.label_encoders['Commodity'].transform([commodity])[0]
        except:
            raise ValueError("Unknown state/district/market/commodity. Please check your inputs.")

        # Extract date features
        year = pred_date.year
        month = pred_date.month
        day = pred_date.day
        day_of_week = pred_date.dayofweek
        week_of_year = pred_date.isocalendar()[1]

        # Get historical data for this combination
        hist = self.historical_data[
            (self.historical_data['State'] == state_enc) &
            (self.historical_data['District'] == district_enc) &
            (self.historical_data['Market'] == market_enc) &
            (self.historical_data['Commodity'] == commodity_enc)
        ].sort_values('Arrival_Date')

        if len(hist) == 0:
            print("Warning: No historical data found for this combination. Using dataset averages.")
            hist = self.historical_data.copy()

        # Calculate lag and MA features
        recent_prices = hist['Modal_Price'].tail(30).values

        modal_price_lag_1 = recent_prices[-1] if len(recent_prices) >= 1 else hist['Modal_Price'].mean()
        modal_price_lag_7 = recent_prices[-7] if len(recent_prices) >= 7 else hist['Modal_Price'].mean()
        modal_price_lag_30 = recent_prices[0] if len(recent_prices) >= 30 else hist['Modal_Price'].mean()
        modal_price_ma_7 = recent_prices[-7:].mean() if len(recent_prices) >= 7 else hist['Modal_Price'].mean()
        modal_price_ma_30 = recent_prices.mean() if len(recent_prices) > 0 else hist['Modal_Price'].mean()

        # Get min and max price estimates
        min_price = hist['Min_Price'].tail(30).mean()
        max_price = hist['Max_Price'].tail(30).mean()

        if model_type.lower() == 'xgboost':
            # Create feature vector
            features = pd.DataFrame({
                'State': [state_enc],
                'District': [district_enc],
                'Market': [market_enc],
                'Commodity': [commodity_enc],
                'Min_Price': [min_price],
                'Max_Price': [max_price],
                'Year': [year],
                'Month': [month],
                'Day': [day],
                'Day_of_Week': [day_of_week],
                'Week_of_Year': [week_of_year],
                'Modal_Price_Lag_1': [modal_price_lag_1],
                'Modal_Price_Lag_7': [modal_price_lag_7],
                'Modal_Price_Lag_30': [modal_price_lag_30],
                'Modal_Price_MA_7': [modal_price_ma_7],
                'Modal_Price_MA_30': [modal_price_ma_30]
            })

            # Make prediction
            prediction = self.xgb_predictor.model.predict(features)[0]

        else:  # Prophet
            # Create combo_id
            combo_id = f"{state_enc}_{district_enc}_{market_enc}_{commodity_enc}"

            # Try Prophet prediction
            prediction = self.prophet_predictor.predict(
                combo_id, pred_date, min_price, max_price, month, day_of_week
            )

            if prediction is None:
                print(f"Warning: No Prophet model found for this combination. Using moving average fallback.")
                prediction = modal_price_ma_7

        print(f"\n{'='*60}")
        print(f"PREDICTED MODAL PRICE: ₹{prediction:.2f}")
        print(f"{'='*60}\n")

        return prediction

# ============================================================================
# PART 6: MAIN EXECUTION PIPELINE
# ============================================================================

def main():
    """Main execution function"""

    print("="*80)
    print(" "*20 + "AGRICULTURAL PRICE PREDICTION SYSTEM")
    print("="*80)

    # Step 1: Load and preprocess data
    preprocessor = DataPreprocessor()
    df = preprocessor.load_and_preprocess('/content/dataset.csv')
    preprocessor.save_encoders()

    # Step 2: Train XGBoost model
    xgb_model = XGBoostPricePredictor()
    xgb_model.train(df)
    xgb_model.save_model()

    # Step 3: Train Prophet model (optional - can be skipped if too slow)
    train_prophet = input("\nTrain Prophet model? (y/n) [Recommended - takes 5-10 min]: ").lower() == 'y'

    if train_prophet:
        prophet_model = ProphetPricePredictor()
        prophet_model.train(df, max_models=50)
        prophet_model.save_model()

    # Step 4: Initialize prediction system
    print("\n" + "="*80)
    print("INITIALIZING PREDICTION SYSTEM")
    print("="*80)

    pred_system = PricePredictionSystem()
    pred_system.load_models()
    pred_system.set_historical_data(df)

    # Step 5: Interactive prediction
    print("\n" + "="*80)
    print("READY FOR PREDICTIONS!")
    print("="*80)

    return pred_system, df

# Run the pipeline
if __name__ == "__main__":
    pred_system, df = main()

    # Example predictions
    print("\n\n" + "="*80)
    print("EXAMPLE PREDICTIONS")
    print("="*80)

    # Get sample values from dataset
    sample = df.iloc[100]
    state = pred_system.preprocessor.label_encoders['State'].inverse_transform([int(sample['State'])])[0]
    district = pred_system.preprocessor.label_encoders['District'].inverse_transform([int(sample['District'])])[0]
    market = pred_system.preprocessor.label_encoders['Market'].inverse_transform([int(sample['Market'])])[0]
    commodity = pred_system.preprocessor.label_encoders['Commodity'].inverse_transform([int(sample['Commodity'])])[0]

    # Make prediction with XGBoost
    print("\n1. XGBoost Prediction:")
    predicted_price_xgb = pred_system.predict(
        state=state,
        district=district,
        market=market,
        commodity=commodity,
        date='2024-12-01',
        model_type='xgboost'
    )

    # Make prediction with Prophet (if trained)
    if len(pred_system.prophet_predictor.models) > 0:
        print("\n2. Prophet Prediction:")
        predicted_price_prophet = pred_system.predict(
            state=state,
            district=district,
            market=market,
            commodity=commodity,
            date='2024-12-01',
            model_type='prophet'
        )

# ============================================================================
# PART 7: SIMPLE PREDICTION FUNCTION FOR EXTERNAL USE
# ============================================================================

def quick_predict(state, district, market, commodity, date, model_type='xgboost'):
    """
    Quick prediction function for external use

    Usage:
    ------
    price = quick_predict('Maharashtra', 'Pune', 'Pune Market', 'Tomato', '2024-12-01')
    price = quick_predict('Maharashtra', 'Pune', 'Pune Market', 'Tomato', '2024-12-01', 'prophet')
    """
    if 'pred_system' not in globals():
        print("Loading models...")
        system = PricePredictionSystem()
        system.load_models()
        # Need to load historical data
        preprocessor = DataPreprocessor()
        df = preprocessor.load_and_preprocess('/content/dataset.csv')
        system.set_historical_data(df)
        globals()['pred_system'] = system

    return pred_system.predict(state, district, market, commodity, date, model_type)

print("\n" + "="*80)
print("PIPELINE COMPLETE!")
print("="*80)
print("\nYou can now make predictions using:")
print("pred_system.predict(state, district, market, commodity, date, model_type='xgboost')")
print("OR")
print("pred_system.predict(state, district, market, commodity, date, model_type='prophet')")
print("\nOr use the quick function:")
print("quick_predict('State', 'District', 'Market', 'Commodity', 'YYYY-MM-DD', 'xgboost')")
print("quick_predict('State', 'District', 'Market', 'Commodity', 'YYYY-MM-DD', 'prophet')")
print("="*80)

                    AGRICULTURAL PRICE PREDICTION SYSTEM
Loading dataset...
Processing dates...
Encoding categorical variables...
Imputing missing values...
Creating time-series features...
Preprocessing complete! Dataset shape: (200000, 19)
Label encoders saved to label_encoders.pkl

Training XGBoost Model
Training on 160000 samples...

XGBoost Model Performance:
MAE: 82.64
RMSE: 225.20
R² Score: 0.9473

XGBoost model saved to xgboost_model.pkl

Train Prophet model? (y/n) [Recommended - takes 5-10 min]: y

Training Prophet Models


DEBUG:cmdstanpy:input tempfile: /tmp/tmpv0iyn4er/2whl1p4f.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpv0iyn4er/lfv3ytif.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=91699', 'data', 'file=/tmp/tmpv0iyn4er/2whl1p4f.json', 'init=/tmp/tmpv0iyn4er/lfv3ytif.json', 'output', 'file=/tmp/tmpv0iyn4er/prophet_modelumftl5d7/prophet_model-20251003042844.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
04:28:44 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing


Training models for top 50 commodity-market combinations...

[1/50] Training model for: 4_66_244_3 (611 records)


04:28:44 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
DEBUG:cmdstanpy:input tempfile: /tmp/tmpv0iyn4er/qylngqk_.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpv0iyn4er/ulpf6n_1.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=61092', 'data', 'file=/tmp/tmpv0iyn4er/qylngqk_.json', 'init=/tmp/tmpv0iyn4er/ulpf6n_1.json', 'output', 'file=/tmp/tmpv0iyn4er/prophet_model0i9r9g8q/prophet_model-20251003042844.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
04:28:44 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
04:28:45 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


  ✓ MAE: 15.68

[2/50] Training model for: 13_448_1715_2 (515 records)
  ✓ MAE: 199.27

[3/50] Training model for: 31_235_796_2 (456 records)


...

  ✓ MAE: 1.97

[48/50] Training model for: 12_377_1384_2 (239 records)


04:28:57 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
DEBUG:cmdstanpy:input tempfile: /tmp/tmpv0iyn4er/23pz8ykr.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpv0iyn4er/_24_4lis.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=69787', 'data', 'file=/tmp/tmpv0iyn4er/23pz8ykr.json', 'init=/tmp/tmpv0iyn4er/_24_4lis.json', 'output', 'file=/tmp/tmpv0iyn4er/prophet_model_ct_9q99/prophet_model-20251003042857.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
04:28:57 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
04:28:57 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


  ✓ MAE: 1.20

[49/50] Training model for: 14_7_915_0 (229 records)
  ✓ MAE: 10.82

[50/50] Training model for: 3_56_700_2 (228 records)


DEBUG:cmdstanpy:input tempfile: /tmp/tmpv0iyn4er/19b0ski8.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpv0iyn4er/kk4lby69.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=32456', 'data', 'file=/tmp/tmpv0iyn4er/19b0ski8.json', 'init=/tmp/tmpv0iyn4er/kk4lby69.json', 'output', 'file=/tmp/tmpv0iyn4er/prophet_modeltdvvd5ev/prophet_model-20251003042857.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
04:28:57 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
04:28:57 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


  ✓ MAE: 51.51

Overall Prophet Model Performance:
Models trained: 50
MAE: 79.70
RMSE: 188.02
R² Score: 0.9630

Prophet models saved to prophet_models.pkl

INITIALIZING PREDICTION SYSTEM
Label encoders loaded from label_encoders.pkl
XGBoost model loaded from xgboost_model.pkl
Prophet models loaded from prophet_models.pkl (50 models)

All models loaded successfully!

READY FOR PREDICTIONS!


EXAMPLE PREDICTIONS

1. XGBoost Prediction:

Making Prediction using XGBOOST
State: Karnataka
District: Kalburgi
Market: Kalburgi
Commodity: Wheat
Date: 2024-12-01


PREDICTED MODAL PRICE: ₹1413.30


2. Prophet Prediction:

Making Prediction using PROPHET
State: Karnataka
District: Kalburgi
Market: Kalburgi
Commodity: Wheat
Date: 2024-12-01


PREDICTED MODAL PRICE: ₹1383.57


PIPELINE COMPLETE!

You can now make predictions using:
pred_system.predict(state, district, market, commodity, date, model_type='xgboost')
OR
pred_system.predict(state, district, market, commodity, date, model_type='prophet')
